In [1]:
!find /kaggle/input -maxdepth 3

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/saibrindae
/kaggle/input/datasets/saibrindae/airport-baggage


In [2]:
!cp -r /kaggle/input/datasets/saibrindae/airport-baggage/CLCXray_YOLO /kaggle/working/

In [3]:
# ---------------------------------------------------------
# 5. Fix data.yaml's path (it still points at your local Windows folder)
# ---------------------------------------------------------
from pathlib import Path
import yaml

yaml_path = Path("/kaggle/working/CLCXray_YOLO/data.yaml")
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = "/kaggle/working/CLCXray_YOLO"
with open(yaml_path, "w") as f:
    yaml.dump(cfg, f)

In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.9 MB/s eta 0:00:00


In [5]:
# ---------------------------------------------------------
# 7. Build YOLO12n from scratch — architecture only, no pretrained weights
# ---------------------------------------------------------
from ultralytics import YOLO

model = YOLO("yolo12n.yaml")
print(model.ckpt)   # should print None

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
{}


In [6]:
# ---------------------------------------------------------
# 8. Train
# ---------------------------------------------------------
results = model.train(
    data=str(yaml_path),

    # -------------------------
    # TRAINING
    # -------------------------
    epochs=200,
    patience=9,        # was 7 -- too tight for from-scratch training, which
                         # plateaus temporarily before continuing to improve

    imgsz=640,
    batch=16,

    device=0,
    workers=2,

    # -------------------------
    # FROM SCRATCH
    # -------------------------
    pretrained=False,

    # -------------------------
    # ONLINE AUGMENTATION
    # -------------------------
    fliplr=0.5,
    flipud=0.0,

    degrees=10.0,
    translate=0.05,
    scale=0.3,          # was 0.1 -- your images range ~373x200 to 1280x720,
                         # a bit more scale jitter should help it generalize
                         # across that range (default is 0.5; this splits
                         # the difference rather than going all the way)
    auto_augment=None,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    # -------------------------
    # DISABLE AGGRESSIVE
    # -------------------------
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    perspective=0.0,
    shear=0.0,

    # -------------------------
    # OUTPUT
    # -------------------------
    project="/kaggle/working/YOLO12n_CLCXray",
    name="minority_augmented",
    exist_ok=True,

    save=True,
    plots=True,

    seed=0,
    deterministic=True,
)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/CLCXray_YOLO/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.yaml, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=minority_augmented, nbs=64, nms=None, opset=None, optimi

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/200      4.56G      2.748      4.163      3.786         34        640: 100% ━━━━━━━━━━━━ 250/250 5.4it/s 45.9s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 4.7it/s 6.3s0.2s
                   all        956       2317      0.388      0.153     0.0578     0.0155

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/200      4.56G      2.084       2.82      2.757         25        640: 100% ━━━━━━━━━━━━ 250/250 5.7it/s 44.2s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 30/30 4.6it/s 6.6s0.2s
                   all        956       2317      0.479      0.427      0.297      0.126

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/200      4.56G      1.819      2.095      2.372         28        640: 100% ━━━━━━━━━━━━ 250/250 5.7it/s 44.2s0.2ss
                 Class     Ima

In [7]:
results = model.predict(source="/kaggle/working/CLCXray_YOLO/images/test", conf=0.25, save=False)

import torch
from ultralytics.utils.metrics import box_iou

IOU_THRESH = 0.5
matched_ious = []

for r in results:
    gt_path = str(r.path).replace("images", "labels").replace(".png", ".txt")
    gt_boxes, gt_classes = [], []
    with open(gt_path) as f:
        for line in f:
            cid, xc, yc, w, h = map(float, line.split())
            W, H = r.orig_shape[1], r.orig_shape[0]
            gt_boxes.append([(xc-w/2)*W, (yc-h/2)*H, (xc+w/2)*W, (yc+h/2)*H])
            gt_classes.append(int(cid))

    if r.boxes is None or len(r.boxes) == 0 or not gt_boxes:
        continue

    gt_boxes_t = torch.tensor(gt_boxes)
    gt_classes_t = torch.tensor(gt_classes)
    pred_boxes = r.boxes.xyxy.cpu()
    pred_classes = r.boxes.cls.cpu().int()
    ious = box_iou(pred_boxes, gt_boxes_t)

    matched_gt = set()
    for i in range(len(pred_boxes)):
        same_class = gt_classes_t == pred_classes[i]
        candidate = ious[i].clone()
        candidate[~same_class] = 0          # only match within the same class
        best_iou, best_idx = candidate.max(0)
        best_idx = best_idx.item()
        if best_iou >= IOU_THRESH and best_idx not in matched_gt:
            matched_ious.append(best_iou.item())
            matched_gt.add(best_idx)         # each GT box can only be matched once

print(f"Mean IoU ({len(matched_ious)} class-matched detections):",
      sum(matched_ious) / len(matched_ious) if matched_ious else 0.0)


image 1/956 /kaggle/working/CLCXray_YOLO/images/test/05_20_27_625.png: 640x608 1 VacuumCup, 12.1ms
image 2/956 /kaggle/working/CLCXray_YOLO/images/test/05_24_05_796.png: 640x480 1 SwissArmyKnife, 1 PlasticBottle, 67.9ms
image 3/956 /kaggle/working/CLCXray_YOLO/images/test/05_24_40_875.png: 640x640 1 CartonDrinks, 62.4ms
image 4/956 /kaggle/working/CLCXray_YOLO/images/test/05_25_25_703.png: 640x640 2 PlasticBottles, 3 VacuumCups, 1 SprayCans, 14.2ms
image 5/956 /kaggle/working/CLCXray_YOLO/images/test/05_27_15_406.png: 640x640 1 VacuumCup, 13.9ms
image 6/956 /kaggle/working/CLCXray_YOLO/images/test/05_29_34_453.png: 352x640 1 VacuumCup, 64.5ms
image 7/956 /kaggle/working/CLCXray_YOLO/images/test/05_34_07_015.png: 416x640 2 PlasticBottles, 2 SprayCanss, 62.3ms
image 8/956 /kaggle/working/CLCXray_YOLO/images/test/05_34_26_437.png: 640x512 2 PlasticBottles, 64.5ms
image 9/956 /kaggle/working/CLCXray_YOLO/images/test/05_36_50_031.png: 640x544 (no detections), 62.3ms
image 10/956 /kaggle/wo

In [8]:
# ---------------------------------------------------------
# 10. GFLOPs and parameter count -- required by the assignment
# ---------------------------------------------------------
model.info(detailed=True)

layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d     False         432       [16, 3, 3, 3]  -0.00841     0.452        float32
    1                       model.0.bn.weight         BatchNorm2d     False          16                [16]      0.98     0.101        float32
    1                         model.0.bn.bias         BatchNorm2d     False          16                [16]     0.117     0.488        float32
    2                             model.0.act                SiLU     False           0                  []         -         -              -
    3                     model.1.conv.weight              Conv2d     False        4608      [32, 16, 3, 3]  -0.00241     0.161        float32
    4                       model.1.bn.weight         BatchNorm2d     False          32                [32]      1.08     0.201        float32
    4         

(272, 2570388, 0, 7.4722304)

In [9]:
# ---------------------------------------------------------
# 11. Generate prediction images 
# ---------------------------------------------------------
model.predict(
    source="/kaggle/working/CLCXray_YOLO/images/test",
    save=True,
    conf=0.25,
    project="/kaggle/working/YOLO12n_CLCXray",
    name="test_predictions",
)


image 1/956 /kaggle/working/CLCXray_YOLO/images/test/05_20_27_625.png: 640x608 1 VacuumCup, 16.6ms
image 2/956 /kaggle/working/CLCXray_YOLO/images/test/05_24_05_796.png: 640x480 1 SwissArmyKnife, 1 PlasticBottle, 16.5ms
image 3/956 /kaggle/working/CLCXray_YOLO/images/test/05_24_40_875.png: 640x640 1 CartonDrinks, 14.9ms
image 4/956 /kaggle/working/CLCXray_YOLO/images/test/05_25_25_703.png: 640x640 2 PlasticBottles, 3 VacuumCups, 1 SprayCans, 14.0ms
image 5/956 /kaggle/working/CLCXray_YOLO/images/test/05_27_15_406.png: 640x640 1 VacuumCup, 13.4ms
image 6/956 /kaggle/working/CLCXray_YOLO/images/test/05_29_34_453.png: 352x640 1 VacuumCup, 14.5ms
image 7/956 /kaggle/working/CLCXray_YOLO/images/test/05_34_07_015.png: 416x640 2 PlasticBottles, 2 SprayCanss, 14.6ms
image 8/956 /kaggle/working/CLCXray_YOLO/images/test/05_34_26_437.png: 640x512 2 PlasticBottles, 14.4ms
image 9/956 /kaggle/working/CLCXray_YOLO/images/test/05_36_50_031.png: 640x544 (no detections), 14.0ms
image 10/956 /kaggle/wo

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 depth: None
 keypoints: None
 masks: None
 names: {0: 'blade', 1: 'scissors', 2: 'knife', 3: 'dagger', 4: 'SwissArmyKnife', 5: 'PlasticBottle', 6: 'Cans', 7: 'VacuumCup', 8: 'GlassBottle', 9: 'CartonDrinks', 10: 'Tin', 11: 'SprayCans'}
 obb: None
 orig_img: array([[[241, 238, 162],
         [245, 240, 167],
         [241, 239, 160],
         ...,
         [242, 235, 169],
         [243, 238, 165],
         [244, 242, 161]],
 
        [[248, 240, 171],
         [244, 237, 169],
         [241, 238, 162],
         ...,
         [244, 244, 157],
         [242, 237, 166],
         [242, 237, 165]],
 
        [[240, 232, 171],
         [239, 231, 171],
         [241, 230, 176],
         ...,
         [246, 235, 176],
         [247, 236, 177],
         [245, 238, 169]],
 
        ...,
 
        [[236, 237, 153],
         [239, 227, 178],
         [241, 239, 161],
         ...,
      

In [10]:
# ============================================================
# FINAL TEST SET EVALUATION
# ============================================================

import numpy as np
import pandas as pd
from ultralytics import YOLO

# ------------------------------------------------------------
# Load best model
# ------------------------------------------------------------
model = YOLO(
    "/kaggle/working/YOLO12n_CLCXray/minority_augmented/weights/best.pt"
)



# ------------------------------------------------------------
# Evaluate on TEST set
# ------------------------------------------------------------
metrics = model.val(
    data=str(yaml_path),
    split="test"
)


# ------------------------------------------------------------
# Class names
# ------------------------------------------------------------
class_names = list(model.names.values())


# ------------------------------------------------------------
# Precision, Recall and F1
# ------------------------------------------------------------
precision = metrics.box.p
recall = metrics.box.r

f1 = np.divide(
    2 * precision * recall,
    precision + recall,
    out=np.zeros_like(precision),
    where=(precision + recall) != 0
)


# ------------------------------------------------------------
# Per-class results
# ------------------------------------------------------------
per_class_df = pd.DataFrame({
    "class": class_names,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "mAP50": metrics.box.ap50,
    "mAP50-95": metrics.box.ap,
})


# ------------------------------------------------------------
# Display per-class results
# ------------------------------------------------------------
print("\nPER-CLASS TEST RESULTS")
print("=" * 70)

print(
    per_class_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# Overall results
# ------------------------------------------------------------
overall_precision = metrics.box.mp
overall_recall = metrics.box.mr
overall_map50 = metrics.box.map50
overall_map50_95 = metrics.box.map

overall_f1 = (
    2 * overall_precision * overall_recall /
    (overall_precision + overall_recall)
    if (overall_precision + overall_recall) > 0
    else 0
)


print("\nOVERALL TEST RESULTS")
print("=" * 70)
print(f"Precision   : {overall_precision:.4f} ({overall_precision*100:.2f}%)")
print(f"Recall      : {overall_recall:.4f} ({overall_recall*100:.2f}%)")
print(f"F1-score    : {overall_f1:.4f} ({overall_f1*100:.2f}%)")
print(f"mAP@50      : {overall_map50:.4f} ({overall_map50*100:.2f}%)")
print(f"mAP@50-95   : {overall_map50_95:.4f} ({overall_map50_95*100:.2f}%)")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO12n summary (fused): 159 layers, 2,559,068 parameters, 0 gradients, 7.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5072.6±760.8 MB/s, size: 590.5 KB)
val: Scanning /kaggle/working/CLCXray_YOLO/labels/test... 956 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 956/956 202.4it/s 4.7s0.0s
val: New cache created: /kaggle/working/CLCXray_YOLO/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 6.1it/s 9.8s0.1s
                   all        956       1421      0.786      0.608      0.643       0.51
                 blade         27         34      0.824      0.676      0.662      0.465
              scissors         51         54      0.983      0.704      0.717      0.591
                 knife         16         16          1      0.971      0.995      0.843
                dagger         19         19     

In [11]:
!zip -r my_output.zip /kaggle/working/YOLO12n_CLCXray
from IPython.display import FileLink
FileLink(r'my_output.zip')

  adding: kaggle/working/YOLO12n_CLCXray/ (stored 0%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/ (stored 0%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/BoxF1_curve.png (deflated 5%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/weights/ (stored 0%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/weights/last.pt (deflated 11%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/weights/best.pt (deflated 11%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/train_batch2.jpg (deflated 13%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/train_batch0.jpg (deflated 13%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/val_batch1_pred.jpg (deflated 11%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/val_batch2_labels.jpg (deflated 11%)
  adding: kaggle/working/YOLO12n_CLCXray/minority_augmented/val_batch0_labels.jpg (deflated 17%)
  adding: kaggle/working/YOLO12n_CLCXray/minori

/kaggle/working/my_output.zip